In [3]:
from dotenv import load_dotenv
from pprint import pprint
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

load_dotenv()

True

## Creating subagents

In [4]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [9]:
# create subagents

subagent_1 = create_agent(
    name="subagent_1",
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    name="subagent_2",
    model='gpt-5-nano',
    tools=[square]
)

## Calling subagents

In [10]:
@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    name="main_agent",
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [11]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [12]:
pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='b9559536-8a4d-4edc-bbf1-0a90c1693439'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2463, 'prompt_tokens': 202, 'total_tokens': 2665, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2432, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWSiVzJPCfs3hxMqdComUpgMOUB4b', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='main_agent', id='lc_run--019da75b-08a4-7f42-867c-b81aeffcaf8b-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_DSM2zR4trlD0CULvZmECqSLr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'